# 🔬 Reference-Free 2D Class Averaging in Cryo-EM

---

## Overview

After particles are picked from micrographs, single-particle cryo-EM
images are still extremely noisy — each raw particle image typically has
**SNR well below 1**, meaning individual particles are barely
interpretable by eye. **2D class averaging** groups particles that share
the same (or similar) viewing orientation, aligns them to each other,
and averages within each group. Averaging $N$ *aligned* noisy copies of
the same signal suppresses independent noise by a factor of
$\sqrt{N}$, turning unrecognizable single particles into
crisp, interpretable 2D projections — while simultaneously giving a
first look at how many distinct orientations/conformations are present.

This notebook builds a **reference-free rotational alignment + class
averaging pipeline** on **real published cryo-EM micrographs**:

| Module | Topic |
|--------|-------|
| **1**  | Real Micrographs, Binning for SNR, & Particle Extraction |
| **2**  | Why Class Averaging Needs Per-Particle Alignment (Multi-Reference Alignment theory) |
| **3**  | Vectorized Rotational Alignment + Iterative Class-Average Update |
| **4**  | Results: Class Averages, Membership, & Quantified Noise Reduction |
| **5**  | Production Methods & Limitations |

### Dataset

We use real **Keyhole Limpet Hemocyanin (KLH)** negative-stain/cryo-EM
micrographs from the classic **NRAMM KLH benchmark dataset** (mirrored
in the `jianlin-cheng/DeepCryoEM` GitHub repository). KLH is a
textbook 2D-classification example because it presents **two
structurally distinct views**: circular "top views" (looking down the
didecamer's symmetry axis) and rectangular "side views" (looking
across it) — exactly the kind of heterogeneity 2D classification is
designed to sort out.

> **Prerequisites:** `numpy`, `scipy`, `scikit-image`, `Pillow`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import os
import glob
import time
import warnings
import urllib.request
from urllib.parse import quote

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import ndimage
from scipy.ndimage import rotate as nd_rotate
from skimage.feature import peak_local_max
from skimage.transform import downscale_local_mean

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — DOWNLOAD REAL KLH MICROGRAPHS (NRAMM benchmark dataset)
# ============================================================
DATA_DIR = "klh_data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = ("https://raw.githubusercontent.com/jianlin-cheng/DeepCryoEM/"
            "master/KLH%20DATASET/")
FILENAMES = [
    "01nov26b.001.001.001.002.png", "01nov26b.001.002.001.002.png", "01nov26b.001.003.001.002.png",
    "01nov26b.001.004.001.002.png", "01nov26b.001.005.001.002.png", "01nov26b.001.006.001.002.png",
    "01nov26b.002.001.001.002.png", "01nov26b.002.002.001.002.png", "01nov26b.002.003.001.002.png",
    "01nov26b.002.004.001.002.png", "01nov26b.002.005.001.002.png", "01nov26b.002.006.001.002.png",
]
for fname in FILENAMES:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(BASE_URL + quote(fname), fpath)
        except Exception as e:
            print(f"  Warning: could not fetch {fname}: {e}")

slice_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.png")))
print(f"Downloaded {len(slice_files)} real KLH micrographs (NRAMM benchmark dataset)")

## 1.1 Binning for Signal-to-Noise

These micrographs are recorded at full camera sampling, which
oversamples the achievable resolution and leaves pixel-level shot
noise dominating the raw image. **Binning** (block-averaging
neighboring pixels) trades spatial resolution for SNR — exactly the
resolution/dose/SNR trade-off every cryo-EM processing pipeline
navigates. We bin 4×4 here purely to make particles visually and
algorithmically tractable for this tutorial.

In [ ]:
# ============================================================
# MODULE 1 — LOAD, BIN, AND PICK PARTICLES
# ============================================================
def load_binned(fpath, bin_factor=4):
    """Load a micrograph, bin it, and percentile-normalize to [0,1]."""
    arr = np.array(Image.open(fpath).convert("L")).astype(np.float32)
    binned = downscale_local_mean(arr, (bin_factor, bin_factor))
    p1, p99 = np.percentile(binned, 1), np.percentile(binned, 99)
    return np.clip((binned - p1) / (p99 - p1), 0, 1)


def pick_particles(img, sigma=3, min_distance=20, pct=97):
    """Simple local-darkness peak detector (particles are dark blobs)."""
    smoothed = ndimage.gaussian_filter(img, sigma=sigma)
    inverted = 1 - smoothed
    return peak_local_max(inverted, min_distance=min_distance,
                           threshold_abs=np.percentile(inverted, pct))


def refine_center(img, y, x, search=8):
    """Recenter a rough pick on its local intensity-weighted centroid."""
    h, w = img.shape
    y, x = int(y), int(x)
    y0, y1 = max(0, y - search), min(h, y + search)
    x0, x1 = max(0, x - search), min(w, x + search)
    patch = img[y0:y1, x0:x1]
    weight = (1 - patch) - (1 - patch).min()
    if weight.sum() < 1e-6:
        return y, x
    ys, xs = np.mgrid[y0:y1, x0:x1]
    return (ys * weight).sum() / weight.sum(), (xs * weight).sum() / weight.sum()


BOX = 64  # particle box size in binned pixels


def extract_box(img, y, x, box=BOX):
    h, w = img.shape
    half = box // 2
    y, x = int(round(y)), int(round(x))
    if y - half < 0 or x - half < 0 or y + half >= h or x + half >= w:
        return None
    return img[y - half:y + half, x - half:x + half]


all_patches = []
for fpath in slice_files:
    img = load_binned(fpath)
    for (y, x) in pick_particles(img):
        cy, cx = refine_center(img, y, x)
        patch = extract_box(img, cy, cx)
        if patch is not None:
            all_patches.append(patch)

particles = np.stack(all_patches)
print(f"Extracted {len(particles)} real particle images "
      f"({BOX}x{BOX} px) from {len(slice_files)} micrographs")

# ------------------------------------------------------------------
# VISUALIZATION 1 — Random sample of raw extracted particles
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
fig.suptitle("Module 1 — Raw Extracted Particles (real KLH data, unsorted)",
             color=ACCENT, fontweight="bold")
sample_idx = np.random.RandomState(3).choice(len(particles), 16, replace=False)
for ax, i in zip(axes.flat, sample_idx):
    ax.imshow(particles[i], cmap="gray")
    ax.axis("off")
plt.tight_layout()
plt.show()

Notice how hard it already is to tell top views (circular) from side
views (rectangular) from single raw particles — this is the noise
problem class averaging exists to solve.

---
# Module 2 — Why Class Averaging Needs Alignment

## 2.1 The Alignment Problem

Two particle images of the *same* orientation will not simply average
well — they were picked at random in-plane rotation and, less
critically for a centered box, random translation. Averaging
*unaligned* copies of the same view destroys the very signal you are
trying to enhance (misaligned edges blur into mush). Every particle
must first be assigned **(a) a class** (which orientation group it
belongs to) **and (b) an alignment** (the rotation, and often
translation, that best matches it to that class) before averaging.

## 2.2 Multi-Reference Alignment (MRA)

This notebook implements a simplified, classical **multi-reference
alignment** scheme (the conceptual ancestor of IMAGIC's MRA/MSA and a
rotation-only special case of what RELION's 2D classification solves
probabilistically):

1. Start with $K$ initial class references (real particle images)
2. For every particle, cross-correlate against every class reference
   **at a grid of rotation angles**, and assign it to the
   (class, angle) pair with highest normalized correlation
3. Recompute each class average from its newly assigned, newly
   rotation-aligned members
4. Repeat until class assignments stabilize

$$\text{corr}(\mathbf{p}, \mathbf{r}_{k,\theta}) = \frac{(\mathbf{p}-\bar{p})\cdot(\mathbf{r}_{k,\theta}-\bar{r})}{\|\mathbf{p}-\bar{p}\|\,\|\mathbf{r}_{k,\theta}-\bar{r}\|}$$

where $\mathbf{r}_{k,\theta}$ is class reference $k$ rotated by angle
$\theta$. This is exactly a normalized cross-correlation — the same
metric underlying the classical template-matching picker from the
particle-picking notebook, now used for orientation search instead of
particle detection.

---
# Module 3 — Vectorized Rotational Alignment & Class-Average Update

The key implementation trick: rotate the **$K$ class references**
(small number) across all candidate angles once per iteration, then
score *every particle against every rotated reference simultaneously*
with a single matrix multiplication — far cheaper than rotating every
particle against every reference individually.

In [ ]:
# ============================================================
# MODULE 3 — REFERENCE-FREE ROTATIONAL MRA / K-MEANS CLASSIFICATION
# ============================================================
def normalize_patch(p):
    """Zero-mean, unit-norm — turns a dot product into a correlation coefficient."""
    p = p - p.mean()
    return p / (np.linalg.norm(p) + 1e-8)


K = 4                       # number of classes
N_ANGLES = 24                # rotation search resolution (15 degree steps)
N_ITERS = 6
angles = np.linspace(0, 360, N_ANGLES, endpoint=False)
N = len(particles)

norm_particles = np.stack([normalize_patch(p) for p in particles])
particles_flat = norm_particles.reshape(N, -1)

rng = np.random.RandomState(0)
class_avgs = particles[rng.choice(N, K, replace=False)].copy()

assignments = np.zeros(N, dtype=int)
best_angles = np.zeros(N)
history = []

t0 = time.time()
for it in range(N_ITERS):
    # --- rotate the K references across all candidate angles (cheap: K*N_ANGLES rotations) ---
    ref_stack, ref_meta = [], []
    for k in range(K):
        for a in angles:
            r = normalize_patch(nd_rotate(class_avgs[k], a, reshape=False, order=1, mode="reflect"))
            ref_stack.append(r.flatten())
            ref_meta.append((k, a))
    ref_mat = np.stack(ref_stack)  # (K*N_ANGLES, BOX*BOX)

    # --- score every particle against every rotated reference in one matmul ---
    corr = particles_flat @ ref_mat.T          # (N, K*N_ANGLES)
    best_idx = np.argmax(corr, axis=1)
    assignments = np.array([ref_meta[i][0] for i in best_idx])
    best_angles = np.array([ref_meta[i][1] for i in best_idx])

    # --- rebuild class averages from newly aligned members (N rotations, not N*K*N_ANGLES) ---
    new_avgs = np.zeros_like(class_avgs)
    counts = np.zeros(K, dtype=int)
    for i in range(N):
        k = assignments[i]
        aligned = nd_rotate(particles[i], -best_angles[i], reshape=False, order=1, mode="reflect")
        new_avgs[k] += aligned
        counts[k] += 1
    for k in range(K):
        if counts[k] > 0:
            new_avgs[k] /= counts[k]
        else:
            new_avgs[k] = class_avgs[k]
    class_avgs = new_avgs
    history.append(counts.copy())
    print(f"iteration {it+1}/{N_ITERS} | class sizes: {counts} | elapsed {time.time()-t0:.1f}s")

print("Alignment + classification complete.")

---
# Module 4 — Results: Class Averages & Quantified Noise Reduction

## 4.1 Expected Signature

Top views are (approximately) rotationally symmetric — our rotation-only
alignment should align them very well, producing sharp, ring-like
averages. Side views are *not* rotationally symmetric in the same way
and also need translational alignment we have not implemented — so we
expect side-view classes to average out noisier and blurrier than
top-view classes. This is a genuine, informative limitation of a
rotation-only MRA, not a bug — and it is exactly the kind of gap real
tools close with full translation+rotation search (Module 5).

In [ ]:
# ============================================================
# MODULE 4 — VISUALIZE CLASS AVERAGES VS RAW MEMBERS
# ============================================================
fig, axes = plt.subplots(2, K, figsize=(4 * K, 8))
fig.suptitle("Module 4 — Class Averages vs. a Raw Member (real KLH particles)",
             color=ACCENT, fontweight="bold")
for k in range(K):
    members = np.where(assignments == k)[0]
    if len(members) > 0:
        axes[0, k].imshow(particles[members[0]], cmap="gray")
    axes[0, k].set_title(f"class {k} — raw example (n={len(members)})", color=TEXT, fontsize=10)
    axes[1, k].imshow(class_avgs[k], cmap="gray")
    axes[1, k].set_title(f"class {k} — average", color=TEXT, fontsize=10)
    for ax in (axes[0, k], axes[1, k]):
        ax.axis("off")
plt.tight_layout()
plt.show()

## 4.2 Quantifying the Noise Reduction

We measure the standard deviation of a **flat background region**
(a corner patch, away from any particle) in a single raw particle
image versus the same region in its class average. If alignment and
averaging are working, background noise should shrink roughly with
$1/\sqrt{N_{\text{class}}}$.

In [ ]:
# ============================================================
# MODULE 4 — QUANTITATIVE NOISE-REDUCTION METRIC
# ============================================================
def corner_std(img, size=14):
    return img[:size, :size].std()


raw_stds, avg_stds, class_sizes = [], [], []
for k in range(K):
    members = np.where(assignments == k)[0]
    if len(members) == 0:
        continue
    raw_stds.append(np.mean([corner_std(particles[m]) for m in members[:20]]))
    avg_stds.append(corner_std(class_avgs[k]))
    class_sizes.append(len(members))

predicted_reduction = [1 / np.sqrt(n) for n in class_sizes]
observed_reduction = [a / r for a, r in zip(avg_stds, raw_stds)]

print("Class sizes:            ", class_sizes)
print("Predicted 1/sqrt(N):    ", [f"{v:.3f}" for v in predicted_reduction])
print("Observed std ratio:     ", [f"{v:.3f}" for v in observed_reduction])

# ------------------------------------------------------------------
# VISUALIZATION 2 — Predicted vs. observed noise reduction
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(class_sizes))
width = 0.35
ax.bar(x - width / 2, predicted_reduction, width, label=r"Predicted $1/\sqrt{N}$", color="#f0883e")
ax.bar(x + width / 2, observed_reduction, width, label="Observed background std ratio", color=ACCENT)
ax.set_xticks(x)
ax.set_xticklabels([f"class {k}\n(n={n})" for k, n in zip(range(K), class_sizes)])
ax.set_ylabel("Background noise ratio (average / raw)")
ax.set_title("Module 4 — Averaging Suppresses Noise Roughly as Predicted",
             color=ACCENT, fontweight="bold")
ax.legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Alignment Model | Notes |
|--------|------------------|-------|
| IMAGIC MRA/MSA (van Heel et al.) | Rotation + translation cross-correlation, multivariate statistical analysis | Classical predecessor to this notebook's approach |
| RELION 2D Classification (Scheres 2012) | Full probabilistic ML/EM over rotation, translation, and class | Marginalizes over alignment rather than taking a single best match |
| cryoSPARC 2D Classification | GPU-accelerated ML/EM, similar formulation to RELION | Widely used in modern pipelines |
| EMAN2 `e2refine2d` | Iterative MSA + K-means, similar spirit to this notebook | Open-source, historically influential |

## Known Limitations of This Tutorial
- **Rotation-only alignment**: no translational search was implemented,
  which is why non-symmetric side-view classes stayed noisier than
  symmetric top-view classes (Module 4).
- **Hard assignment**: each particle is assigned to its single best
  (class, angle); production ML/EM methods marginalize (soft-assign)
  over all possibilities, which is far more robust to noise.
- **$K$ chosen by hand**: real pipelines often over-classify (large
  $K$) and merge or discard classes manually or via hierarchical
  schemes.
- **No CTF correction**: real single-particle pipelines correct for
  the contrast transfer function before or during classification;
  this notebook works directly on binned, uncorrected images.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Reference-Free 2D Class Averaging — Pipeline Dashboard",
             fontsize=16, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.3)

for k in range(min(K, 4)):
    ax = fig.add_subplot(gs[0, k])
    ax.imshow(class_avgs[k], cmap="gray")
    ax.set_title(f"class {k} average (n={class_sizes[k] if k < len(class_sizes) else 0})",
                 color=TEXT, fontsize=10)
    ax.axis("off")

ax4 = fig.add_subplot(gs[1, 0:2])
hist_arr = np.array(history)
for k in range(K):
    ax4.plot(range(1, N_ITERS + 1), hist_arr[:, k], marker="o", label=f"class {k}")
ax4.set_xlabel("Iteration"); ax4.set_ylabel("Class size")
ax4.set_title("Class membership over iterations", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)

ax5 = fig.add_subplot(gs[1, 2])
ax5.bar([f"c{k}" for k in range(len(class_sizes))], class_sizes, color=ACCENT)
ax5.set_title("Final class sizes", color=TEXT, fontsize=10)

ax6 = fig.add_subplot(gs[1, 3])
ax6.bar(x - width / 2, predicted_reduction, width, color="#f0883e", label="predicted")
ax6.bar(x + width / 2, observed_reduction, width, color=ACCENT, label="observed")
ax6.set_title("Noise reduction: pred. vs. obs.", color=TEXT, fontsize=9)
ax6.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=7)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Real data | 1 | Real NRAMM KLH micrographs, binned for SNR, particles picked and centered |
| Problem framing | 2 | Class averaging requires joint class assignment + alignment (Multi-Reference Alignment) |
| Implementation | 3 | Vectorized rotation search: rotate references (cheap), correlate all particles at once (one matmul) |
| Results | 4 | Sharp top-view class averages; noisier side-view classes — an honest, informative limitation |
| Context | 5 | Positioned against IMAGIC, RELION, cryoSPARC, EMAN2 |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Particle picking (Module 1) | $\mathcal{O}(N_{\text{mic}}\cdot P^2)$ | Gaussian smoothing per micrograph |
| Reference rotation (per iter.) | $\mathcal{O}(K \cdot A \cdot B^2)$ | Small: $K$=classes, $A$=angles, $B$=box size |
| Particle scoring (per iter.) | $\mathcal{O}(N \cdot K \cdot A \cdot B^2)$ | Dominant; one matrix multiply |
| Class-average rebuild (per iter.) | $\mathcal{O}(N \cdot B^2)$ | One rotation per particle |

## Key References
- van Heel & Frank (1981) — Multivariate statistical analysis for 2D EM classification
- Scheres (2012) — RELION: Bayesian view of cryo-EM structure determination (*J. Struct. Biol.*)
- Punjani et al. (2017) — cryoSPARC: algorithms for rapid unsupervised cryo-EM structure determination (*Nature Methods*)
- Tang et al. (2007) — EMAN2: extensible image processing suite for electron microscopy
- NRAMM KLH benchmark dataset (source of the micrographs used here, via the DeepCryoPicker GitHub mirror)